# KBS 뉴스 URL 수집 — 카테고리 페이지 Selenium / 기간 모드 (Colab용)

KBS 뉴스의 일자별 카테고리 페이지(`/news/pc/category/category.do?ref=pSiteMap#YYYYMMDD&N`)를 Selenium으로 열어 JS 비동기 렌더링이 끝난 DOM에서 기사 URL을 추출한다. 카테고리 페이지의 일자/페이지는 URL fragment(`#` 뒤)에 들어있어 서버에 전달되지 않으므로 `requests`로 받으면 빈 껍데기만 받는다. 따라서 브라우저 JS가 `/api/getNewsList`를 자동 호출해 채운 DOM을 우리가 읽는 식 — API를 우리 코드가 직접 호출하지 않는다.

통신3사/LPOD/SBS 트랙과 동일하게 **기간 단위 통합 JSON** 1개를 만들고, 내부에서 일자별 페이지를 순회하며 임시 체크포인트로 중단/재개를 지원한다.

- 입력: `press_ranges` — 언론사(press) + 기간(start_date, end_date)
- 출력: `data/링크_{press}_{YYMMDD}_{YYMMDD}.json` (기간 통합 view.do URL 리스트)
- 보조 출력: 기간별 수집 로그 JSON, 중간 재개용 temp JSON
- 특징: Selenium으로 카테고리 페이지 렌더링, 사진뉴스 자동 제외(`exceptPhotoYn='Y'`가 카테고리 페이지 기본값), 일자/페이지 fragment 기반 순회, 일자별 임시 체크포인트
- 본문 단계 호환: 출력 JSON은 `["https://news.kbs.co.kr/news/view.do?ncd=...", ...]` 단순 URL 리스트라 `KBS_직접_본문_수집_bs4_colab.ipynb`가 그대로 입력으로 받음
- 트랙 B(네이버 경유, press='KBS')와 파일명 충돌 방지를 위해 press 이름에 `_direct` 접미사 사용

In [1]:
# Colab 환경 세팅 — Selenium, Google Chrome 설치 (BS4 불필요, DOM은 Selenium으로 직접 읽음)
!wget -q -O /tmp/google-chrome.deb https://dl.google.com/linux/direct/google-chrome-stable_current_amd64.deb
!apt-get install -y -q /tmp/google-chrome.deb
!pip install -q selenium

Reading package lists...
Building dependency tree...
Reading state information...
The following additional packages will be installed:
  at-spi2-core gsettings-desktop-schemas libatk-bridge2.0-0 libatk1.0-0
  libatk1.0-data libatspi2.0-0 libvulkan1 libxcomposite1 libxtst6
  mesa-vulkan-drivers session-migration
The following NEW packages will be installed:
  at-spi2-core google-chrome-stable gsettings-desktop-schemas
  libatk-bridge2.0-0 libatk1.0-0 libatk1.0-data libatspi2.0-0 libvulkan1
  libxcomposite1 libxtst6 mesa-vulkan-drivers session-migration
0 upgraded, 12 newly installed, 0 to remove and 3 not upgraded.
Need to get 11.2 MB/141 MB of archives.
After this operation, 477 MB of additional disk space will be used.
Get:1 http://archive.ubuntu.com/ubuntu jammy/main amd64 libatk1.0-data all 2.36.0-3build1 [2,824 B]
Get:2 http://archive.ubuntu.com/ubuntu jammy/main amd64 libatk1.0-0 amd64 2.36.0-3build1 [51.9 kB]
Get:3 http://archive.ubuntu.com/ubuntu jammy/main amd64 libatspi2.0-0 a

In [2]:
# Google Drive 마운트 — 중간에 끊겨도 데이터 보존
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
# Drive 안의 프로젝트 폴더로 이동
# 통신3사 트리와 분리하기 위해 KBS/SBS 등 언론사 직접 수집은 news/ 하위에 보관
import os
PROJECT_DIR = '/content/drive/MyDrive/Text-data-Analysis_26-Spring/news'
os.chdir(PROJECT_DIR)
print(f'현재 작업 폴더: {os.getcwd()}')

현재 작업 폴더: /content/drive/MyDrive/Text-data-Analysis_26-Spring/news


In [7]:
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
import json
import os
import shutil
import subprocess
import time
from datetime import datetime, timedelta
from pathlib import Path

# 로컬/Colab 비교를 위해 User-Agent 고정
USER_AGENT = 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/147.0.0.0 Safari/537.36'

# 담당 언론사와 수집 기간 지정 — 날짜 형식: 'YYYY.MM.DD'
# press 하나당 start_date ~ end_date 안의 일자를 한 통합 JSON으로 묶어 저장 (통신3사/LPOD/SBS 트랙과 동일 패턴)
# 트랙 B(네이버 경유, press='KBS') 결과와 파일명 충돌 방지를 위해 press 이름에 _direct 접미사 사용
press_ranges = [
    {'press': 'KBS_direct', 'start_date': '2026.05.05', 'end_date': '2026.05.11'},
]


# 기간 단위 작업 목록 생성 — press_ranges 한 항목당 jobs 1개
# 내부 일자 순회는 collect_links_for_period에서 처리하므로 일자별 분할 jobs는 만들지 않음
def build_period_jobs(press_ranges):
    jobs = []
    for item in press_ranges:
        # 'YYYY.MM.DD' 문자열을 datetime으로 파싱 — 기간 유효성 검사용
        start = datetime.strptime(item['start_date'], '%Y.%m.%d')
        end = datetime.strptime(item['end_date'], '%Y.%m.%d')
        # 시작 일자가 끝 일자보다 늦으면 작업 범위가 잘못된 것이므로 즉시 중단
        if start > end:
            raise ValueError(f"시작 일자가 끝 일자보다 늦습니다: {item}")
        jobs.append({
            'press': item['press'],
            'start_date': item['start_date'],
            'end_date': item['end_date'],
        })
    return jobs


# 생성된 jobs는 다음 셀에서 순서대로 실행
jobs = build_period_jobs(press_ranges)

print(f'총 작업 수: {len(jobs)}')
for job in jobs:
    print(job)

# 셀 3을 건너뛰고 실행해도 기본 프로젝트 경로를 사용할 수 있게 보완
try:
    PROJECT_DIR
except NameError:
    PROJECT_DIR = '/content/drive/MyDrive/Text-data-Analysis_26-Spring/news'

# 저장할 폴더 지정 — 링크 파일, 수집 로그, 임시 체크포인트, 실패 목록이 모두 이 폴더에 저장
SAVE_DIR = Path(PROJECT_DIR) / 'notebook' / 'crawling' / 'data'
SAVE_DIR.mkdir(parents=True, exist_ok=True)
print(f'저장 위치: {SAVE_DIR}')

# Chrome 옵션 — 통신3사 url_수집_colab.ipynb와 동일 패턴 (Colab 컨테이너 + headless)
options = Options()
options.add_argument(f'user-agent={USER_AGENT}')  # 요청 환경을 일정하게 유지하기 위해 User-Agent 고정
options.add_experimental_option('excludeSwitches', ['enable-automation'])  # 자동화 제어 관련 switch 제외
options.add_experimental_option('useAutomationExtension', False)  # Selenium 자동화 확장 비활성화
options.add_argument('--disable-blink-features=AutomationControlled')  # AutomationControlled 플래그 비활성화
options.add_argument('--headless=new')  # Colab은 GUI가 없으므로 새 headless 모드 사용
options.add_argument('--no-sandbox')  # Colab 컨테이너 환경에서 Chrome 실행 안정화
options.add_argument('--disable-dev-shm-usage')  # /dev/shm 용량 부족으로 Chrome이 죽는 문제 완화
options.add_argument('--disable-gpu')  # headless 환경에서 GPU 관련 오류 방지
options.add_argument('--window-size=1920,1080')  # headless에서도 일정한 화면 크기로 렌더링

# Colab chromium-browser 패키지는 snap 래퍼라 Selenium에서 자주 실패
# 설치 셀에서 받은 Google Chrome 사용, ChromeDriver는 Selenium Manager에 맡김
chrome_binary = shutil.which('google-chrome') or shutil.which('google-chrome-stable') or '/usr/bin/google-chrome'
if not os.path.exists(chrome_binary):
    raise FileNotFoundError('Google Chrome을 찾지 못했습니다. 설치 셀을 먼저 다시 실행해 주세요.')
options.binary_location = chrome_binary
print(f'Chrome binary: {chrome_binary}')
subprocess.run([chrome_binary, '--version'], check=False)

# Chrome 드라이버 설정
# Selenium Manager가 현재 Chrome 버전에 맞는 ChromeDriver를 자동으로 찾거나 내려받음
driver = webdriver.Chrome(service=Service(), options=options)

# navigator.webdriver 플래그 제거 — 자동화 탐지를 회피하기 위해 새 페이지 진입 시마다 주입
driver.execute_cdp_cmd('Page.addScriptToEvaluateOnNewDocument', {
    'source': 'Object.defineProperty(navigator, "webdriver", {get: () => undefined})'
})
print(f'User-Agent: {USER_AGENT}')

총 작업 수: 1
{'press': 'KBS_direct', 'start_date': '2026.05.05', 'end_date': '2026.05.11'}
저장 위치: /content/drive/MyDrive/Text-data-Analysis_26-Spring/news/notebook/crawling/data
Chrome binary: /usr/bin/google-chrome
User-Agent: Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/147.0.0.0 Safari/537.36


In [8]:
import random
import re
from selenium.common.exceptions import StaleElementReferenceException

# 서버 부담을 줄이기 위해 페이지/일자/job 사이에 랜덤 대기
# PAGE_PAUSE: 같은 일자 내 페이지 이동 (짧게 — JS가 비동기로 API 호출하므로 잠시만 대기)
# DAY_PAUSE: 같은 기간 내 일자 전환 (중간) — 통신3사 트랙과 동일 값
# JOB_PAUSE: 다른 언론사로 전환 (길게)
PAGE_PAUSE_RANGE_SEC = (1, 3)
DAY_PAUSE_RANGE_SEC = (2, 5)
JOB_PAUSE_RANGE_SEC = (5, 12)
# DOM 갱신이 비동기라 WebDriverWait 최대 대기를 넉넉히 (KBS API가 일시 느리면 그만큼 기다림)
SELENIUM_WAIT_SEC = 20
# WebDriverWait poll 간격 — 기본 0.5초보다 길게 잡으면 stale 발생률 감소 (DOM 갱신 중간에 잡힐 확률 ↓)
SELENIUM_POLL_FREQ_SEC = 0.8
SKIP_COMPLETED = True

CATEGORY_PAGE_BASE = 'https://news.kbs.co.kr/news/pc/category/category.do?ref=pSiteMap'
# 기사 view URL은 절대 URL로 정규화 — 본문 노트북(KBS_직접_본문_수집_bs4_colab)이 기대하는 포맷
VIEW_URL_BASE = 'https://news.kbs.co.kr/news/view.do'


# 랜덤 대기 후 로그 출력
def polite_sleep(label, pause_range):
    pause_sec = random.uniform(*pause_range)
    print(f"{label} {pause_sec:.1f}초 대기")
    time.sleep(pause_sec)


# 현재 페이지의 box-content a 태그 href 리스트를 stale-safe하게 추출
# 각 a마다 try/except로 stale 발생 시 그 요소만 건너뜀 (전체 wait 중단되지 않게)
def safe_get_hrefs(driver):
    a_elements = driver.find_elements(By.CSS_SELECTOR, '.category-main-list .box-contents a.box-content')
    hrefs = []
    for a in a_elements:
        try:
            hrefs.append(a.get_attribute('href') or '')
        except StaleElementReferenceException:
            # 이 요소가 stale이면 건너뜀 — 다음 a 처리 계속
            continue
    return hrefs


# 현재 페이지에서 view.do URL set 추출
# href에서 ncd 숫자만 뽑아 정규 URL로 재조립 — 쿼리스트링 변형/추가 파라미터로 인한 중복 차단
def extract_view_urls(driver):
    links = set()
    for href in safe_get_hrefs(driver):
        m = re.search(r'view\.do\?ncd=(\d+)', href)
        if m:
            links.add(f'{VIEW_URL_BASE}?ncd={m.group(1)}')
    return links


# 현재 페이지 첫 box-content href (변경 감지용 — 새 페이지/일자 로드 신호)
def first_href(driver):
    hrefs = safe_get_hrefs(driver)
    return hrefs[0] if hrefs else ''


# 카테고리 페이지 최초 로드 + init() 완료 대기 (일자별 driver.get은 fragment만 다른 같은 URL이라 reload 안 돼 한 번만 호출)
# init() 완료 신호: maxPageNo 변수 정의 + box-contents 채워짐
def initial_load(driver, wait):
    driver.get(CATEGORY_PAGE_BASE)

    def ready(d):
        try:
            has_pagination = d.execute_script('return typeof maxPageNo !== "undefined" && maxPageNo > 0;')
            has_articles = len(d.find_elements(By.CSS_SELECTOR, '.category-main-list .box-contents a.box-content')) > 0
            return bool(has_pagination and has_articles)
        except StaleElementReferenceException:
            return False

    wait.until(ready)


# 일자 변경 — JS searchDate 변수 set 후 goPage() 직접 호출
# driver.get(...#YYYYMMDD&1)은 fragment만 다른 같은 URL이라 브라우저가 reload 안 함 (history만 갱신) → 데이터 그대로 남는 버그
# 대신 KBS inline JS의 흐름을 그대로 흉내: searchDate 갱신 → goPage()가 새 일자 데이터를 API로 가져옴
def goto_date(driver, wait, date_ymd):
    # 변경 전 첫 href 기억 — 새 일자 데이터 도착 감지용
    prev = first_href(driver)
    driver.execute_script(f"""
        searchDate = '{date_ymd}';
        currentPageNo = 1;
        goPage();
    """)

    # 새 일자 데이터 도착 조건: currentPageNo == 1 + box-contents 채워짐 + 첫 href가 이전 일자와 다름
    def ready(d, _prev=prev):
        try:
            if d.execute_script('return currentPageNo;') != 1:
                return False
            hrefs = safe_get_hrefs(d)
            if not hrefs:
                return False
            return hrefs[0] != _prev
        except StaleElementReferenceException:
            return False

    wait.until(ready)


# 같은 일자 안 페이지 N으로 이동 — JS movePage(N) 호출 (fragment 리스너 없어서 location.hash로 안 됨)
def goto_page(driver, wait, page):
    prev = first_href(driver)
    driver.execute_script(f'movePage({page});')

    # 새 페이지 데이터 도착 조건: currentPageNo == page + box-contents 채워짐 + 첫 href가 이전 페이지와 다름
    def ready(d, _page=page, _prev=prev):
        try:
            if d.execute_script('return currentPageNo;') != _page:
                return False
            hrefs = safe_get_hrefs(d)
            if not hrefs:
                return False
            return hrefs[0] != _prev
        except StaleElementReferenceException:
            return False

    wait.until(ready)


# JS의 maxPageNo 변수 직접 읽음 — setPagination()이 일자 변경 시마다 갱신해 줌
def read_max_page(driver):
    return int(driver.execute_script('return maxPageNo;'))


# 한 일자의 모든 페이지를 순회하며 기사 URL set 반환 + 페이지별 통계
# 호출 직전에 카테고리 페이지가 이미 로드되어 있어야 함 (collect_links_for_period의 initial_load 책임)
def collect_links_for_day(driver, date_ymd):
    # WebDriverWait — ignored_exceptions로 stale 예외 무시하고 재시도 + poll 간격 늘려 stale 발생률 감소
    wait = WebDriverWait(
        driver, SELENIUM_WAIT_SEC,
        poll_frequency=SELENIUM_POLL_FREQ_SEC,
        ignored_exceptions=(StaleElementReferenceException,),
    )

    # 일자 변경 + 1페이지 데이터 도착 대기
    goto_date(driver, wait, date_ymd)
    max_page = read_max_page(driver)

    day_links = set()
    page_stats = []

    # 1 ~ max_page 순회 — 1페이지는 이미 도착해 있음
    for page in range(1, max_page + 1):
        if page > 1:
            # 페이지 이동 전 짧은 대기 (서버 페이스 조절)
            polite_sleep(f"  page {page} 이동 전", PAGE_PAUSE_RANGE_SEC)
            goto_page(driver, wait, page)

        # DOM에서 URL 추출 (stale-safe)
        page_links = extract_view_urls(driver)
        before = len(day_links)
        day_links.update(page_links)
        # 페이지별 수집량 로그 — 특정 페이지에서 0건이면 셀렉터 변경 의심
        page_stats.append({
            'page': page,
            'found': len(page_links),
            'added': len(day_links) - before,
            'total_this_day': len(day_links),
        })

    return day_links, max_page, page_stats


# 한 언론사의 기간 전체를 통합 JSON 1개로 저장 (통신3사 collect_links와 동일 패턴)
# 일자별 임시 체크포인트로 중단/재개 지원
def collect_links_for_period(press, start_date, end_date, driver=None, save_dir=SAVE_DIR):
    # 파일명에 들어가는 기간 접미사 (예: 2026.05.01~2026.05.07 -> 260501_260507)
    start_yymmdd = start_date.replace('.', '')[2:]
    end_yymmdd = end_date.replace('.', '')[2:]
    period = f"{start_yymmdd}_{end_yymmdd}"

    # 파일 경로 — temp: 일자 단위 체크포인트 / links: 최종 통합 / stats: 수집 로그
    temp_links_path = save_dir / f"{press}_{start_date}_{end_date}_temp_links.json"
    links_save_path = save_dir / f"링크_{press}_{period}.json"
    stats_save_path = save_dir / f"수집로그_{press}_{period}.json"

    # 최종 파일이 이미 있으면 같은 기간은 건너뜀 (밤새 재시작에도 idempotent)
    if SKIP_COMPLETED and links_save_path.exists():
        print()
        print(f"=== {press} / {start_date} ~ {end_date} 이미 완료됨, 건너뜀 ===")
        print(f"기존 파일: {links_save_path}")
        return links_save_path

    print()
    print(f"=== {press} / {start_date} ~ {end_date} 수집 시작 ===")

    # 임시 파일에 기존 링크가 있으면 불러오기 — last_date 다음 날부터 이어서 수집
    if temp_links_path.exists():
        with temp_links_path.open('r', encoding='utf-8') as f:
            checkpoint = json.load(f)
        # set으로 변환해 이미 모은 링크와 신규 링크 중복 차단
        all_links_set = set(checkpoint.get('links', []))
        last_collected_date = checkpoint.get('last_date')
        # daily_stats도 임시 파일에 같이 보관해서 중단 전 로그 유지
        daily_stats = checkpoint.get('daily_stats', [])
        print(f"기존 임시 파일에서 링크 {len(all_links_set)}개 불러옴 — {last_collected_date} 다음부터 이어서 수집")
    else:
        all_links_set = set()
        last_collected_date = None
        daily_stats = []
        print('새로 링크 수집 시작')

    # 카테고리 페이지를 한 번만 로드 — 이후 일자 변경은 searchDate + goPage()
    initial_wait = WebDriverWait(
        driver, SELENIUM_WAIT_SEC,
        poll_frequency=SELENIUM_POLL_FREQ_SEC,
        ignored_exceptions=(StaleElementReferenceException,),
    )
    initial_load(driver, initial_wait)

    # 시작 ~ 끝 일자를 하루씩 순회
    current = datetime.strptime(start_date, '%Y.%m.%d')
    end = datetime.strptime(end_date, '%Y.%m.%d')

    while current <= end:
        day_str = current.strftime('%Y.%m.%d')

        # 이미 수집 완료한 날짜면 건너뜀 (재시작 시 중복 fetch 차단)
        if last_collected_date and day_str <= last_collected_date:
            print(f"{day_str} — 이미 수집 완료, 건너뜀")
            current += timedelta(days=1)
            continue

        date_ymd = day_str.replace('.', '')  # 'YYYYMMDD' — searchDate JS 변수에 set
        started_at = time.time()

        # 하루치 모든 페이지 순회해서 URL 추출
        day_links, max_page, page_stats = collect_links_for_day(driver, date_ymd)
        before = len(all_links_set)
        all_links_set.update(day_links)
        added = len(all_links_set) - before  # 일자 간 중복 제거 후 늘어난 개수
        elapsed = round(time.time() - started_at, 2)

        # 일자별 수집량/페이지 수/소요 시간 로그 — 사후 진단용
        daily_stats.append({
            'date': day_str,
            'unique_in_day': len(day_links),
            'added': added,
            'total': len(all_links_set),
            'max_page': max_page,
            'elapsed_sec': elapsed,
            'pages': page_stats,
        })

        print(
            f"{day_str} — 일자 내 unique {len(day_links)}건 / 신규 {added}건 "
            f"/ 누적 {len(all_links_set)}건 / 페이지 {max_page} / {elapsed}초"
        )

        # 하루치 수집 후 임시 파일에 즉시 저장 (중간에 끊겨도 누적 보존 + 마지막 완료 날짜 기록)
        with temp_links_path.open('w', encoding='utf-8') as f:
            json.dump(
                {'links': sorted(all_links_set), 'last_date': day_str, 'daily_stats': daily_stats},
                f, ensure_ascii=False, indent=2,
            )
        last_collected_date = day_str
        current += timedelta(days=1)
        # 다음 날짜로 넘어가기 전 대기 (마지막 날 뒤엔 안 함)
        if current <= end:
            polite_sleep('다음 날짜 전', DAY_PAUSE_RANGE_SEC)

    # 기간 통합 최종 저장 — sorted로 안정적 순서 (diff/재현성)
    kbs_news_links = sorted(all_links_set)
    with open(links_save_path, 'w', encoding='utf-8') as f:
        json.dump(kbs_news_links, f, ensure_ascii=False, indent=2)

    # 수집 로그 — 일자별 통계 + 전체 요약
    with open(stats_save_path, 'w', encoding='utf-8') as f:
        json.dump({
            'press': press,
            'start_date': start_date,
            'end_date': end_date,
            'total': len(kbs_news_links),
            'days': daily_stats,
        }, f, ensure_ascii=False, indent=2)

    # 정상 완료 시 임시 파일 삭제 — 다음 실행에서 그릇된 재개 방지
    if temp_links_path.exists():
        temp_links_path.unlink()

    print(f"수집 완료 — 총 {len(kbs_news_links)}개")
    print(f"링크 저장: {links_save_path}")
    print(f"수집 로그 저장: {stats_save_path}")
    return links_save_path


# job 단위로 실행 (job 하나 = 언론사 × 기간)
# 한 작업이 실패해도 실패 목록에 기록하고 다음 작업으로 넘어감
results = []
failures = []
for index, job in enumerate(jobs, start=1):
    print()
    print(f"[{index}/{len(jobs)}] 작업 실행: {job}")
    try:
        # job 딕셔너리의 press/start_date/end_date를 collect_links_for_period 인자로 전달
        results.append(collect_links_for_period(**job, driver=driver))
    except Exception as exc:
        # 한 언론사에서 오류가 나도 전체 작업이 멈추지 않도록 실패 정보만 저장
        failures.append({'job': job, 'error': repr(exc)})
        print(f"작업 실패, 다음 작업으로 넘어갑니다: {exc!r}")
    finally:
        if index < len(jobs):
            # 다음 job(언론사)으로 넘어가기 전 대기
            polite_sleep('다음 작업 전', JOB_PAUSE_RANGE_SEC)

# 실패한 작업이 있으면 나중에 다시 돌릴 수 있게 파일로 저장
if failures:
    failures_path = SAVE_DIR / '수집실패목록_KBS_direct.json'
    with open(failures_path, 'w', encoding='utf-8') as f:
        json.dump(failures, f, ensure_ascii=False, indent=2)
    print()
    print(f"실패 작업 {len(failures)}개 저장: {failures_path}")

print()
print('전체 작업 완료')
print(f'성공/건너뜀: {len(results)}개, 실패: {len(failures)}개')
for result_path in results:
    print(result_path)



[1/1] 작업 실행: {'press': 'KBS_direct', 'start_date': '2026.05.05', 'end_date': '2026.05.11'}

=== KBS_direct / 2026.05.05 ~ 2026.05.11 수집 시작 ===
기존 임시 파일에서 링크 1345개 불러옴 — 2026.05.07 다음부터 이어서 수집
2026.05.05 — 이미 수집 완료, 건너뜀
2026.05.06 — 이미 수집 완료, 건너뜀
2026.05.07 — 이미 수집 완료, 건너뜀
  page 2 이동 전 2.1초 대기
  page 3 이동 전 2.7초 대기
  page 4 이동 전 2.8초 대기
  page 5 이동 전 2.3초 대기
  page 6 이동 전 1.1초 대기
  page 7 이동 전 2.2초 대기
  page 8 이동 전 1.6초 대기
  page 9 이동 전 1.8초 대기
  page 10 이동 전 1.1초 대기
  page 11 이동 전 2.1초 대기
  page 12 이동 전 1.1초 대기
  page 13 이동 전 1.3초 대기
  page 14 이동 전 1.4초 대기
  page 15 이동 전 2.3초 대기
  page 16 이동 전 2.3초 대기
  page 17 이동 전 1.7초 대기
  page 18 이동 전 1.1초 대기
  page 19 이동 전 1.1초 대기
  page 20 이동 전 1.8초 대기
  page 21 이동 전 1.4초 대기
  page 22 이동 전 1.6초 대기
  page 23 이동 전 2.9초 대기
  page 24 이동 전 1.6초 대기
  page 25 이동 전 1.6초 대기
  page 26 이동 전 2.5초 대기
  page 27 이동 전 1.5초 대기
  page 28 이동 전 2.7초 대기
  page 29 이동 전 2.1초 대기
  page 30 이동 전 2.1초 대기
  page 31 이동 전 1.5초 대기
  page 32 이동 전 2.4초 대기
  page 33 이동 전 1.6초 대

In [6]:
# 브라우저 창 닫기 (필요하면 이 셀만 따로 실행)
try:
    driver.quit()
    print('브라우저 종료 완료')
except Exception as exc:
    print(f'브라우저 종료 중 오류: {exc!r}')

브라우저 종료 완료
